# Разбор `LexicalFunction` из DISSECT

Этот ноутбук объясняет код из `dissect-master/src/composes/composition/lexical_function.py` по частям.

**Цель класса:** обучить линейные функции вида `p = U * v`, где:
- `v` — вектор аргумента,
- `U` — матрица функционального слова,
- `p` — вектор составной конструкции.

## Часть 1. Импорты и объявление класса

Здесь подключаются базовые зависимости: матрицы, регрессия, утилиты приведения типов и базовый класс `CompositionModel`.

In [ ]:
import numpy as np
import time
from composition_model import CompositionModel
from composes.semantic_space.space import Space
from composes.utils.gen_utils import get_partitions
from composes.utils.gen_utils import assert_valid_kwargs
from composes.utils.regression_learner import LstsqRegressionLearner
from composes.utils.regression_learner import RegressionLearner
from composes.utils.matrix_utils import resolve_type_conflict
from composes.utils.matrix_utils import get_type_of_largest
from composes.utils.matrix_utils import padd_matrix
from composes.utils.num_utils import is_integer
from composes.utils.gen_utils import assert_is_instance
from composes.exception.illegal_state_error import IllegalStateError

class LexicalFunction(CompositionModel):
    _name = "lexical_function"
    _MIN_SAMPLES = 1

## Часть 2. Конструктор `__init__`

**Что делает:**
1. Проверяет допустимые параметры.
2. Ставит learner по умолчанию (`LstsqRegressionLearner`).
3. Либо принимает готовый `function_space`, либо learner для обучения.
4. Хранит флаг intercept.

In [ ]:
def __init__(self, **kwargs):
    assert_valid_kwargs(kwargs, ["function_space", "intercept", "learner"])

    self._regression_learner = LstsqRegressionLearner()
    self.composed_id2column = []
    self._function_space = None
    self._has_intercept = False

    if "function_space" in kwargs:
        space = kwargs["function_space"]
        if not isinstance(space, Space):
            raise TypeError("expected Space-type argument")
        self._function_space = kwargs["function_space"]

    if "intercept" in kwargs:
        has_intercept = kwargs["intercept"]
        if not isinstance(has_intercept, bool):
            raise TypeError("expected bool-type argument")
        self._has_intercept = has_intercept

    if "learner" in kwargs:
        if "function_space" in kwargs:
            raise ValueError("cannot instantiate with both learner and function_space!")
        self._regression_learner = kwargs["learner"]

## Часть 3. Обучение `train`

**Вход:** тройки `(function_word, arg, phrase)`.

**Логика:**
1. Сортировка и фильтрация валидных строк.
2. Группировка по `function_word`.
3. Для каждой группы обучение регрессии `arg -> phrase`.
4. Преобразование коэффициентов регрессии в матрицу функции.
5. Сбор всех матриц в `self._function_space`.

In [ ]:
def train(self, train_data, arg_space, phrase_space):
    self._has_intercept = self._regression_learner.has_intercept()

    if not isinstance(arg_space, Space):
        raise ValueError("expected one input spaces!")

    result_mats = []
    train_data = sorted(train_data, key=lambda tup: tup[0])

    function_word_list, arg_list, phrase_list = self.valid_data_to_lists(
        train_data,
        (None, arg_space.row2id, phrase_space.row2id)
    )

    keys, key_ranges = get_partitions(function_word_list, self._MIN_SAMPLES)
    if not keys:
        raise ValueError("No valid training data found!")

    assert len(arg_space.element_shape) == 1

    if self._has_intercept:
        new_element_shape = phrase_space.element_shape + (arg_space.element_shape[0] + 1,)
    else:
        new_element_shape = phrase_space.element_shape + (arg_space.element_shape[0],)

    for i in xrange(len(key_ranges)):
        idx_beg, idx_end = key_ranges[i]
        arg_mat = arg_space.get_rows(arg_list[idx_beg:idx_end])
        phrase_mat = phrase_space.get_rows(phrase_list[idx_beg:idx_end])

        matrix_type = get_type_of_largest([arg_mat, phrase_mat])
        [arg_mat, phrase_mat] = resolve_type_conflict([arg_mat, phrase_mat], matrix_type)

        # Learner возвращает матрицу регрессии; transpose приводит к форме функции U
        result_mat = self._regression_learner.train(arg_mat, phrase_mat).transpose()
        result_mat.reshape((1, np.prod(new_element_shape)))
        result_mats.append(result_mat)

    new_space_mat = arg_mat.nary_vstack(result_mats)
    self.composed_id2column = phrase_space.id2column
    self._function_space = Space(new_space_mat, keys, [], element_shape=new_element_shape)

## Часть 4. Применение модели `compose`

**Что делает:**
1. Для каждой пары `(function_word, arg)` берет матрицу функции и вектор аргумента.
2. Вызывает `_compose` (матрично-векторное умножение).
3. Склеивает все предсказанные векторы в новый `Space`.

In [ ]:
def compose(self, data, arg_space):
    assert_is_instance(arg_space, Space)

    arg1_list, arg2_list, phrase_list = self.valid_data_to_lists(
        data,
        (self._function_space.row2id, arg_space.row2id, None)
    )

    composed_vec_list = []
    for i in xrange(len(arg1_list)):
        arg1_vec = self._function_space.get_row(arg1_list[i])
        arg2_vec = arg_space.get_row(arg2_list[i])

        matrix_type = get_type_of_largest([arg1_vec, arg2_vec])
        [arg1_vec, arg2_vec] = resolve_type_conflict([arg1_vec, arg2_vec], matrix_type)

        composed_ph_vec = self._compose(arg1_vec, arg2_vec, self._function_space.element_shape)
        composed_vec_list.append(composed_ph_vec)

    result_element_shape = self._function_space.element_shape[0:-1]
    composed_ph_mat = composed_ph_vec.nary_vstack(composed_vec_list)

    return Space(composed_ph_mat, phrase_list, self.composed_id2column, element_shape=result_element_shape)

## Часть 5. Внутренняя операция `_compose`

Ключевая математика: преобразовать плоское представление матрицы функции в `U`, затем посчитать `U * v`.

Если включен intercept, к `v` добавляется константный признак.

In [ ]:
def _compose(self, function_arg_vec, arg_vec, function_arg_element_shape):
    new_shape = (
        np.prod(function_arg_element_shape[0:-1]),
        function_arg_element_shape[-1]
    )

    function_arg_vec.reshape(new_shape)

    if self._has_intercept:
        comp_el = function_arg_vec * padd_matrix(arg_vec.transpose(), 0)
    else:
        comp_el = function_arg_vec * arg_vec.transpose()

    return comp_el.transpose()

## Часть 6. Связь с вашей задачей про сербские приставки

Интерпретация полей в вашем кейсе:
- `function_word` -> приставка,
- `arg` -> базовый глагол,
- `phrase` -> глагол с приставкой.

Тогда `train` учит отдельную матрицу на каждую приставку, а `compose` предсказывает вектор производного глагола.

## Часть 7. Важные технические замечания

1. В коде используется `xrange`, то есть он из эпохи Python 2.
2. В современных окружениях обычно заменяют `xrange` на `range`.
3. `_assert_space_match` в исходнике оставлен пустым (`pass`).
4. `MIN_SAMPLES` в коде = 1, хотя в комментарии встречается старое описание про 3.